In [2]:

# E = cleaned_E
from docplex.mp.model import Model
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 150
MinUtilTime_2 = 150
MaxUtilTime = 1000

vehicle_fixed_cost = 1000
freq_penalty = 1000

nvehicle = 150
K = range(nvehicle)
H = range(HOURS)

bigM = 1440

freq1 = [0]*17 + [3, 4, 5, 4, 2, 0, 0]  # ATB
freq2 = [0]*17 + [3, 4, 5, 4, 4, 0, 0]  # HSK

arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {("ATB", "HSK"): 110, ("HSK", "ATB"): 110}

# --- NODES ---
nodes = []
node_id = 1
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
start_node_id = node_id
node_id += 1

for hour in H:
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1
    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'})
end_node_id = node_id
node_id += 1

nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

# --- INITIAL ARCS ---
E = []
synthetic_id = node_id
synthetic_nodes = []

for node in nodes:
    loc = node['loc']
    dst_time = node['time']
    if loc in ['ATB', 'HSK']:
        cost, N_bus = arc_data[('X', loc)]
        if dst_time >= cost:
            E.append({
                'src': {'id': start_node_id, 'time': 0, 'loc': 'X'},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })

for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time + travel_time[(curr_loc, transitions[curr_loc])] <= MinUtilTime_2:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > MINUTES_IN_DAY:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_nodes.append(dst)

        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1
        })

        curr_src_id = synthetic_id
        curr_time = next_time
        curr_loc = next_loc
        synthetic_id += 1
        total_time += cost

    if curr_time <= MINUTES_IN_DAY:
        cost_to_X, N_bus_to_X = arc_data[(curr_loc, 'X')]
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'},
            'cost': cost_to_X,
            'N_bus': N_bus_to_X
        })

V.extend(synthetic_nodes)

for node in V:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# --- ADD WAITING AND DEADHEADING ARCS ---
MAX_WAIT = 60
augmented_E = E.copy()
unique_keys = set()

# Add existing arcs to unique_keys to avoid duplication later
for e in E:
    arc_key = (
        e['src']['id'], e['src']['time'], e['src']['loc'],
        e['dst']['id'], e['dst']['time'], e['dst']['loc'],
        e['cost'], e['N_bus']
    )
    unique_keys.add(arc_key)

# Waiting arcs
for src in V:
    for dst in V:
        if src['id'] == dst['id']:
            continue
        time_diff = dst['time'] - src['time']
        if src['loc'] == dst['loc'] and 0 < time_diff <= MAX_WAIT:
            arc_key = (
                src['id'], src['time'], src['loc'],
                dst['id'], dst['time'], dst['loc'],
                200, 150
            )
            if arc_key not in unique_keys:
                unique_keys.add(arc_key)
                augmented_E.append({
                    'src': src,
                    'dst': dst,
                    'cost': 200,
                    'N_bus': 150
                })

# Deadheading arcs
# for src in V:
#     for dst in V:
#         if src['id'] == dst['id'] or src['loc'] == dst['loc']:
#             continue

#         key = (src['loc'], dst['loc'])
#         if key not in arc_data:
#             continue

#         travel_t, cap = arc_data[key]
#         if dst['time'] > src['time'] + travel_t:
#             arc_key = (
#                 src['id'], src['time'], src['loc'],
#                 dst['id'], dst['time'], dst['loc'],
#                 travel_t, cap
#             )
#             if arc_key not in unique_keys:
#                 unique_keys.add(arc_key)
#                 augmented_E.append({
#                     'src': src,
#                     'dst': dst,
#                     'cost': 220,
#                     'N_bus': 150
#                 })
for src in V:
    best_arcs_by_dst_loc = {}
    
    for dst in V:
        if src['id'] == dst['id'] or src['loc'] == dst['loc']:
            continue

        key = (src['loc'], dst['loc'])
        if key not in arc_data:
            continue

        travel_t, cap = arc_data[key]

        if dst['time'] > src['time'] + travel_t:
            time_diff = dst['time'] - src['time'] - travel_t

            # Keep only the best arc (least time diff) per dst location
            if dst['loc'] not in best_arcs_by_dst_loc or time_diff < best_arcs_by_dst_loc[dst['loc']]['time_diff']:
                best_arcs_by_dst_loc[dst['loc']] = {
                    'dst': dst,
                    'travel_t': travel_t,
                    'cap': cap,
                    'time_diff': time_diff
                }

    # Now select the single best arc among all dst_locs (least time_diff)
    if best_arcs_by_dst_loc:
        # Find arc with minimal time_diff overall
        best_dst_loc = min(best_arcs_by_dst_loc, key=lambda loc: best_arcs_by_dst_loc[loc]['time_diff'])
        best = best_arcs_by_dst_loc[best_dst_loc]

        arc_key = (
            src['id'], src['time'], src['loc'],
            best['dst']['id'], best['dst']['time'], best['dst']['loc'],
            best['travel_t'], best['cap']
        )

        if arc_key not in unique_keys:
            unique_keys.add(arc_key)
            augmented_E.append({
                'src': src,
                'dst': best['dst'],
                'cost': 220,
                'N_bus': 150
            })


# Final cleaned arcs
E = augmented_E


# --- FINAL OUTPUT [unchanged] ---
for arc in E:
    print({
        'src_id': arc['src']['id'],
        'src_loc': arc['src']['loc'],
        'src_time': arc['src']['time'],
        'dst_id': arc['dst']['id'],
        'dst_loc': arc['dst']['loc'],
        'dst_time': arc['dst']['time'],
        'cost': arc['cost'],
        'N_bus': arc['N_bus']
    })
# --- MODEL ---
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

node_ids_in_arcs = set()
for e in E:
    node_ids_in_arcs.add(e['src']['id'])
    node_ids_in_arcs.add(e['dst']['id'])

arrival_time = mdl.continuous_var_dict(((k, nid) for k in K for nid in node_ids_in_arcs), name='arr', lb=0, ub=MINUTES_IN_DAY)
freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)

mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K) +
    mdl.sum(vehicle_fixed_cost * z[k] for k in K) 
)
    # mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)

for k in K:
    mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E))

    mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['src']['id'] == start_node_id) == z[k])
    mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['dst']['id'] == end_node_id) == z[k])

    for node in V:
        i = node['id']
        if i != start_node_id and i != end_node_id:
            mdl.add_constraint(
                mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['src']['id'] == i) ==
                mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['dst']['id'] == i)
            )

for e in E:
    if e['N_bus'] == 1:
        mdl.add_constraint(mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for k in K) <= 1)

for k in K:
    mdl.add_constraint(T[k] == mdl.sum(e['cost'] * x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['N_bus'] == 1))
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

for e in E:
    for k in K:
        i, j = e['src']['id'], e['dst']['id']
        cost = e['cost']
        mdl.add_constraint(arrival_time[k, j] >= arrival_time[k, i] + cost - bigM * (1 - x.get((i, j, k), 0)))
        mdl.add_constraint(arrival_time[k, j] <= arrival_time[k, i] + cost + bigM * (1 - x.get((i, j, k), 0)))

# for h in H:
#     mdl.add_constraint(
#         mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0)
#                 for k in K
#                 for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h)
#         + freq_slack_ATB[h] >= freq1[h]
#     )
#     mdl.add_constraint(
#         mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0)
#                 for k in K
#                 for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h)
#         + freq_slack_HSK[h] >= freq2[h]
#     )
for h in H:
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h
            for k in K
        ) >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h
            for k in K
        ) >= freq2[h]
    )
solution = mdl.solve(log_output=True)

if solution:
    print("Objective:", solution.objective_value)
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")
        current_node = start_node_id
        while current_node != end_node_id:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x.get((e['src']['id'], e['dst']['id'], k), 0).solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break
            arc = next_arcs[0]
            src, dst = arc['src'], arc['dst']
            src_h, src_m = divmod(int(src['time']), 60)
            dst_h, dst_m = divmod(int(dst['time']), 60)
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")
            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")

{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 2, 'dst_loc': 'ATB', 'dst_time': 1020, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 5, 'dst_loc': 'HSK', 'dst_time': 1020, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 3, 'dst_loc': 'ATB', 'dst_time': 1040, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 6, 'dst_loc': 'HSK', 'dst_time': 1040, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 4, 'dst_loc': 'ATB', 'dst_time': 1060, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 7, 'dst_loc': 'HSK', 'dst_time': 1060, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 8, 'dst_loc': 'ATB', 'dst_time': 1080, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 12, 'dst_loc': 'HSK', 'dst_time': 1080, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time